<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code/06_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF   # 2-4에서 사용

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [2]:
DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy(float)
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from IPython.display import display

# ---------------------------------------------------------
# 1. 데이터 로드 및 파생변수 강제 생성 (KeyError 원천 차단)
# ---------------------------------------------------------
DATA = 'knhanes_like_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy(float)
ST = df['kstrata'].to_numpy()
PS = df['psu'].to_numpy()

# 'male' 변수가 없으면 원본 'sex' 변수를 이용해 자동으로 생성합니다.
if 'male' not in df.columns:
    df['male'] = (df['sex'] == 1).astype(float)


# ---------------------------------------------------------
# 2. 필수 기반 함수 정의 (다른 셀 실행 여부와 무관하게 작동)
# ---------------------------------------------------------
def _vcl(U, strata, psu):
    t = pd.DataFrame(U)
    t['st'] = strata
    t['cl'] = psu
    uc = t.groupby(['st', 'cl']).sum(numeric_only=True)
    p = U.shape[1]
    B = np.zeros((p, p))
    for s, g in uc.groupby(level=0):
        m = len(g)
        if m > 1:
            D = g.values - g.values.mean(axis=0)
            B += (m / (m - 1)) * (D.T @ D)
    return B

def svyglm(y, X, w, strata, psu, family='binomial'):
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    w = np.asarray(w, float)
    if family == 'binomial':
        m = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues
        v  = mu * (1 - mu)
    else:
        m = sm.GLM(y, X, family=sm.families.Gaussian(), freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues
        v  = np.ones_like(mu)
    b = m.params
    A = (X.T * (w * v)) @ X
    U = X * (w * (y - mu))[:, None]
    B = _vcl(U, strata, psu)
    Ai = np.linalg.inv(A)
    V  = Ai @ B @ Ai
    return b, np.sqrt(np.diag(V)), m

def get_or_ci(b, se, idx):
    o = np.exp(b[idx])
    lo = np.exp(b[idx] - 1.96 * se[idx])
    hi = np.exp(b[idx] + 1.96 * se[idx])
    return f"{o:.3f} ({lo:.3f}-{hi:.3f})"


# ---------------------------------------------------------
# 3. B-2 분석: 결측치 제거 및 복합표본 회귀분석
# ---------------------------------------------------------
adj_vars = ['age', 'male', 'edu']
vars_to_use = ['htn', 'drink'] + adj_vars

# 분석할 변수들에 결측치가 있는 행 제거
df_b2 = df.dropna(subset=vars_to_use).copy()

# 인덱스 불일치 오류를 막기 위해 df.loc을 사용하여 추출
y_b2  = df_b2['htn'].astype(float)
w_b2  = df.loc[df_b2.index, 'wt'].to_numpy(float)
ST_b2 = df.loc[df_b2.index, 'kstrata'].to_numpy()
PS_b2 = df.loc[df_b2.index, 'psu'].to_numpy()

# (1) 무보정 모형 적합
X_unadj = sm.add_constant(df_b2[['drink']].astype(float))
b_unadj, se_unadj, _ = svyglm(y_b2, X_unadj, w_b2, ST_b2, PS_b2, family='binomial')

# (2) 보정 모형 적합
X_adj = sm.add_constant(df_b2[['drink'] + adj_vars].astype(float))
b_adj, se_adj, _ = svyglm(y_b2, X_adj, w_b2, ST_b2, PS_b2, family='binomial')


# ---------------------------------------------------------
# 4. Table 출력
# ---------------------------------------------------------
print("=== [B-2] 고위험음주가 고혈압에 미치는 영향 (무보정 vs 보정) ===")
data_res = [
    ["고위험음주(drink)", get_or_ci(b_unadj, se_unadj, 1), get_or_ci(b_adj, se_adj, 1)]
]

for i, col in enumerate(adj_vars, start=2):
    data_res.append([col, "-", get_or_ci(b_adj, se_adj, i)])

df_result = pd.DataFrame(data_res, columns=["변수", "무보정 OR (95% CI)", "보정 OR (95% CI)"])
display(df_result)

=== [B-2] 고위험음주가 고혈압에 미치는 영향 (무보정 vs 보정) ===


,변수,무보정 OR (95% CI),보정 OR (95% CI)
0,고위험음주(drink),1.056 (0.964-1.158),1.113 (0.999-1.239)
1,age,-,1.075 (1.069-1.080)
2,male,-,0.906 (0.781-1.050)
3,edu,-,0.963 (0.903-1.028)
